# Complement Validation - Recovery Eval (IteraTeR clean insertions)

The decisive test: does the learned edge `G(A,B)` recover the gold inserted span?
Baseline ladder: edge vs arith(B-A) vs enc_B vs enc_AB vs enc_A(floor).

**Success = edge beats arith(B-A) AND enc_B** on acc@1 / MRR / mean_cos,
with enc_A near the floor.

Eval uses the FIRST `N_EVAL` triples of training's shuffled pool = training's
**validation slice** (used only for val_loss, never gradient-trained) — held out.
`collapse_sim` is an unreliable proxy; THIS recovery table is the real verdict.

Needs `complement_wiki_best.pt` in Drive (from train_kaggle.ipynb)."

In [ ]:
REPO_URL        = "https://github.com/haaaaaaayrewugrhyw/multihop-retrieval.git"
REPO_NAME       = "multihop-retrieval"
WORK_DIR        = f"/kaggle/working/{REPO_NAME}/complement_wiki"
DRIVE_FOLDER_ID = "1mMCf_lhYcw3CH_ttOWDLgSKZuPYqh5m5"
N_EVAL          = 2000     # must be <= train val_examples (4000) to stay held-out
N_DISTRACT      = 19
POOL            = 124000   # = train max_examples+val (reproduces training's shuffled pool;
                           # eval uses the FIRST N_EVAL = training's val slice, never trained on)

In [ ]:
# 1. GPU + clone + deps
import os, torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Settings -> Accelerator -> T4 GPU")
root = f"/kaggle/working/{REPO_NAME}"
os.system(f"git clone {REPO_URL} {root}" if not os.path.isdir(root) else f"cd {root} && git pull")
os.chdir(WORK_DIR)
os.system("pip install -q 'transformers>=4.35.0' 'datasets>=2.14.0' gdown")
print(torch.cuda.get_device_name(0), "| cwd", os.getcwd())

In [ ]:
# 2. Get checkpoint from Drive (data loads from HuggingFace in the eval script)
import os, sys, gdown, shutil
os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)
sys.path.insert(0, f"{WORK_DIR}/../retrieval_v3")
sys.path.insert(0, f"{WORK_DIR}/../retrieval_v2")
os.makedirs(f"{WORK_DIR}/models", exist_ok=True)
ckpt = f"{WORK_DIR}/models/complement_wiki_best.pt"
if not os.path.exists(ckpt):
    dl = "/kaggle/working/drive_data"
    if not os.path.isdir(dl):
        gdown.download_folder(id=DRIVE_FOLDER_ID, output=dl, quiet=True, use_cookies=False)
    src = f"{dl}/complement_wiki_best.pt"
    if os.path.exists(src):
        shutil.copy(src, ckpt); print("checkpoint copied from Drive")
    else:
        print("WARNING: complement_wiki_best.pt not in Drive - train first")
else:
    print("checkpoint present")

In [ ]:
# 3. Run recovery eval (load via explicit path - bulletproof)
import importlib.util, sys, os
os.chdir(WORK_DIR)
for m in ["eval_recovery", "data_wikiedits", "generator_train"]:
    sys.modules.pop(m, None)
spec = importlib.util.spec_from_file_location("eval_recovery", f"{WORK_DIR}/eval_recovery.py")
eer  = importlib.util.module_from_spec(spec)
sys.modules["eval_recovery"] = eer
sys.argv = ["eval_recovery.py", "--n", str(N_EVAL), "--n_distract", str(N_DISTRACT), "--pool", str(POOL)]
spec.loader.exec_module(eer)
results = eer.main()